In [9]:
import os
import json
import pandas as pd
from termcolor import colored, cprint
from datasets import load_dataset

from inspect_ai.log import read_eval_log

In [10]:
root_dir = "../outputs/"

model_info = pd.read_csv(os.path.join("..", "data", "models_pricing.csv"))
valids_qids = load_dataset("matsant01/blind-spots-bench", split="test").to_pandas()["QID"].tolist()

In [11]:
def capability_to_qtype(model_type: str):
	if model_type == "VLM":
		return ["text", "multi"]
	elif model_type == "LM" or model_type == "LLM":
		return ["text"]
	elif model_type == "GEN":
		return ["image_gen"]
	else:
		raise ValueError(f"Unknown model type: {model_type}")

In [12]:
results = []

for model_name, model_type in zip(model_info["model_name"], model_info["model_type"]):
	cleaned_model_name = model_name.split("/")[1] if "/" in model_name else model_name
	qtypes = capability_to_qtype(model_type)
	
	in_tks_price = model_info[model_info["model_name"] == model_name]["input_price"].iloc[0]
	out_tks_price = model_info[model_info["model_name"] == model_name]["output_price"].iloc[0]

	# Iterate for all folders under outputs/
	for epoch_dir in os.listdir(os.path.join(root_dir)):
		epoch_subdirs = [os.path.join(root_dir, epoch_dir, qtype) for qtype in qtypes]
		
		# In each outputs/<epoch>/ iterate over all question types
		for qtype_subdir in epoch_subdirs:
			if not os.path.exists(qtype_subdir):
				continue

			# Search for the folders matching the model name
			model_dirs = [d for d in os.listdir(qtype_subdir) if d.split("_")[0] == cleaned_model_name]
			if len(model_dirs) == 0:
				cprint(f"No folder found for model {model_name} in {qtype_subdir}", "yellow")
				continue
			elif len(model_dirs) > 1:
				cprint(f"Multiple folders found for model {model_name} in {qtype_subdir}: {model_dirs}", "yellow")
				continue
			model_dir = model_dirs[0]

			# Find the eval log in the model folder
			eval_log_paths = [os.path.join(qtype_subdir, model_dir, f) for f in os.listdir(os.path.join(qtype_subdir, model_dir)) if f.endswith(".eval")]
			if len(eval_log_paths) == 0:
				cprint(f"No eval log found for model {model_name} in {os.path.join(qtype_subdir, model_dir)}", "yellow")
				continue
			elif len(eval_log_paths) > 1:
				cprint(f"Multiple eval logs found for model {model_name} in {os.path.join(qtype_subdir, model_dir)}: {eval_log_paths}", "yellow")
				continue

			# Read the eval log
			eval_log_path = eval_log_paths[0]
			eval_data = read_eval_log(eval_log_path)

			tmp_results = []
			for sample in eval_data.samples:
				qid = sample.id
				# NOTE: apply here the QID filtering!
				if qid not in valids_qids:
					continue


				if qtype_subdir.split("/")[-1] != "image_gen":
					if sample.scores and "exceeded message limit" in sample.scores["model_graded_qa_with_reasoning_stripped"].explanation:
						# ⚠️ NOTE: here we're removing samples where the grader wasn't able to grade. In later steps, we'll add back these samples
						# by taking them from the re-grading output. This is because such samples are not necessarily wrong (as solver-side errors).
						continue

					scores_list = list(sample.scores.values())
					if len(scores_list) == 0:
						score = 0
						error = True
					elif len(scores_list) > 1:
						raise ValueError(f"Multiple scores found for sample {qid} in model {model_name}: {scores_list}")
					else:
						score = 1 if scores_list[0] is not None and scores_list[0].value == "C" and sample.error is None else 0
						error = sample.error is not None

					epoch = int(sample.epoch)

					if not error:
						if len(eval_data.stats.model_usage.keys()) == 1:
							usage = sample.role_usage.get("solver", None)

						elif len(eval_data.stats.model_usage.keys()) > 1:
							sample.model_usage.pop("google/gemini-3-flash-preview", None)
							usage = list(sample.model_usage.values())
							if len(usage) == 0:
								usage = None
							else: 
								usage = usage[0]
						else:
							usage = None
						
						in_tks = usage.input_tokens if usage else 0
						out_tks = usage.output_tokens if usage else 0
						if usage and usage.reasoning_tokens is not None and not "gpt-5" in model_dir.lower():	# NOTE: gpt-5 models count reasoning tokens already as part of output tokens
							out_tks += usage.reasoning_tokens

					else:
						in_tks = 0
						out_tks = 0

				else:
					score = int(sample.scores["image_generation_grader"].value == "C")
					error = sample.error is not None
					if "gemini" in model_dir.lower():
						in_tks += sample.store["image_generation_metadata"]["usage"]["prompt_token_count"]
						out_tks += sample.store["image_generation_metadata"]["usage"]["total_token_count"]
					elif "gpt" in model_dir.lower():
						in_tks += sample.store["image_generation_metadata"]["usage"]["input_tokens"] if "image_generation_metadata" in sample.store else 0
						out_tks += sample.store["image_generation_metadata"]["usage"]["output_tokens"] if "image_generation_metadata" in sample.store else 0


				tmp_results.append({
					"model_name": model_name,
					"model_type": model_type,
					"in_tokens": in_tks,
					"out_tokens": out_tks,
					"in_tks_price": in_tks_price,
					"out_tks_price": out_tks_price,
					"price": (in_tks * in_tks_price + out_tks * out_tks_price) / 1_000_000,
					"epoch": epoch,
					"source": qtype_subdir,
					"qtype": qtype_subdir.split("/")[-1],
					"qid": qid,
					"score": score,
					"error": error,
				})

			results.extend(tmp_results)

No folder found for model deepseek-ai/DeepSeek-V4-Pro in ../outputs/reval/text
No folder found for model moonshotai/Kimi-K2.6 in ../outputs/reval/text


In [13]:
df = pd.DataFrame(results)
df.to_csv(os.path.join("..", "data", "processed_results.csv"), index=False)
len(df)

16667

In [14]:
df.columns

Index(['model_name', 'model_type', 'in_tokens', 'out_tokens', 'in_tks_price',
       'out_tks_price', 'price', 'epoch', 'source', 'qtype', 'qid', 'score',
       'error'],
      dtype='object')

## Merge results with re-graded responses

Merge with the samples that required re-grading because of grader failure.

In [15]:
regr_path = "../data/regrade-results/regrading_outputs.jsonl"

with open(regr_path, "r") as f:
	regr_results = [json.loads(line) for line in f]
regr_df = pd.DataFrame(regr_results)
len(regr_df)

147

In [16]:
regr_df.rename(columns={"solver_name": "model_name", "grader_score": "score", "question_type": "qtype"}, inplace=True)
regr_df["qid"] = regr_df["full_sample"].apply(lambda x: x["index"])
regr_df["epoch"] = 5
regr_df["model_type"] = model_info[model_info["model_name"] == regr_df["model_name"].iloc[0]]["model_type"].iloc[0]
regr_df["score"] = regr_df["score"].apply(lambda x: 1 if x == "C" else 0)
regr_df["qtype"] = regr_df["qtype"].apply(lambda x: "text" if x == "text-only" else ("multi" if "multi-to-text" == x else x))
regr_df["source"] = regr_path
regr_df["error"] = False

regr_df["in_tokens"] = regr_df["full_sample"].apply(lambda x: x["in_tokens"] if "in_tokens" in x else 0)
regr_df["out_tokens"] = regr_df["full_sample"].apply(lambda x: x["out_tokens"] if "out_tokens" in x else 0)


regr_df["in_tks_price"] = regr_df["model_name"].apply(lambda x: model_info[model_info["model_name"] == x]["input_price"].iloc[0])
regr_df["out_tks_price"] = regr_df["model_name"].apply(lambda x: model_info[model_info["model_name"] == x]["output_price"].iloc[0])
regr_df["price"] = (regr_df["in_tokens"] * regr_df["in_tks_price"] + regr_df["out_tokens"] * regr_df["out_tks_price"]) / 1_000_000

In [17]:
df = pd.concat([df, regr_df[["model_name", "model_type", "epoch", "qtype", "qid", "score", "source", "error", "in_tokens", "out_tokens", "price", "in_tks_price", "out_tks_price"]]], ignore_index=True)

In [18]:
len(df)

16814

In [19]:
df[df["qtype"] != "image_gen"].groupby(["model_name", "qid"])["score"].mean()[df[df["qtype"] != "image_gen"].groupby(["model_name", "qid"])["score"].count() < 4]

Series([], Name: score, dtype: float64)

In [20]:
# Limit the number of samples per-question to 4 by taking the first 4 samples for each question (if there are more than 4)
df = df.groupby(["model_name", "qid"]).head(4).reset_index(drop=True)

In [21]:
df.to_csv(os.path.join("..", "data", "processed_results_all.csv"), index=False)

In [25]:
# For each model, make sure answers exist for each valid QID (making sure we check on the right question types)
valids_qids = load_dataset("matsant01/blind-spots-bench", split="test").to_pandas()[["QID", "question_type"]]


for model_name in df["model_name"].unique():
	qtypes = capability_to_qtype(model_info[model_info["model_name"] == model_name]["model_type"].iloc[0])
	for qtype in qtypes:
		model_qtype_qids = df[(df["model_name"] == model_name) & (df["qtype"] == qtype)]["qid"].unique()
		expected_qids = valids_qids[valids_qids["question_type"] == qtype]["QID"].tolist()
		missing_qids = set(expected_qids) - set(model_qtype_qids)
		if len(missing_qids) > 0:
			cprint(f"Model {model_name} is missing answers for {len(missing_qids)} {qtype} questions: {missing_qids}", "red")